# Notebook 1: Practical Machine Learning introduction with Python
*Authors: Sara Speelman & Andrei Covaci; 2026-08-10*  

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
np.random.seed(7) # Set the random seed so results are always the same when re-running the notebook
random_state = 7  # Set a random state variable for use in functions that require it, for reproducability


# 0. Introduction

Machine learning (ML) is the study of algorithms allowing computers to perform a task without being explicitly programmed for it but instead by learning from data. In this Notebook, we will tacke our first ML problems with simple models. In later WPOs, we will evolve to more complex models and pipelines (we will for example add feature selection and hyperparameter tuning).

This session is focused on:
- understanding supervised learning
- tackling both regression and classification problems
- using important libraries such as 
    - scikit-learn, 
    - numpy
    - pandas
- applying the correct high level workflow and experimental setup in an ML project
    - global data inspection and data cleaning
    - correct split in training validation and test set
    - exploratory data analysis on the training set
    - training a model on the training set
    - choosing a model with the validation set or through cross validation
    - estimating the performance on unseen test data



## Supervised learning

The most common field of ML algorithms are the **supervised learning** algorithms. Supervised algorithms learn to perform a task by using a dataset of input/output pairs. The output is also known as the **target variable**.

In practice, if we have a dataset of m "training examples" (a pair of input/output), we can represent it as

$(\vec{x^i},y^i)$,

where $i$ is the index of the pair of input/output and ranges over $1, ..., m$, where $m$ is the amount of $(\vec{x},y)$ pairs present in the dataset. $\vec{x}$ is an input and $y$ is an output.

The goal of a supervised algorithm is to find a good hypothesis function $h$ representing the relation $h(\vec{x})=y$. In practice, we cannot hope to find a perfect hypothesis function $h$ such that $h(\vec{x^i})=y^i$ for all i in our dataset and that would also be true for new data samples drawn from a different dataset. Instead, the hypothesis function $h$ only gives an approximation $\hat{y}$ of the true value $y$, so we have $h(\vec{x})=\hat{y}$. Supervised ML algorithms allows us to find a good hypothesis function $h$ that reduces as much as possible the error between the predictions $\hat{y}^i$ and the true values $y^i$.

Another way to represent the dataset $(\vec{x^i},y^i)^{i=1,...,m}$ is to use the matrix form: $(X,\vec{y})$, where $X=(x^i_j)^{i=1,...,m}_{j=1,...,n}$ and $\vec{y}=(y^i)^{i=1,...,m}$. So $X$ is a $m$ x $n$ matrix of $m$ vectors $x^i$ of dimension $n$, and $\vec{y}$ is a vector of dimension $m$.

## Regression vs Classification

Supervised learning problems typically fall into two main categories, depending on the type of the target variable $y$. 

- **Regression**: $y$ is a continuous value ($y \in \mathbb{R}$). As an example application, we will use regression to predict the median housing price of an area from different housing attributes of this area.
- **Classification**: $y$ takes one of a finite set of class labels. The goal is to predict the correct class for each data point. In this notebook, we will use classification to predict the species of a flower from its measured features.

## Useful libraries
- **Scikit-learn**:  Python library offering a great variety of ML algorithms. You will be using this library a lot throughout the course
- **Numpy**: a fundamental Python library for fast numerical computation using efficient multi-dimensional arrays and mathematical operations.
- **Pandas**: a high-level Python library for loading, cleaning, manipulating, and analyzing structured data using tables (DataFrames).

If you are unfamiliar with numpy and/or pandas, we invite you to have a look at notebook-0-preliminaries

# 1. Regression mini project
In this mini project, we use **linear regression** to predict housing prices.

Scikit-learn is a library offering a large collection of ML algorithms and tools to perform a typical ML application. Have a look at this [quick tutorial](https://scikit-learn.org/stable/getting_started.html) to get started, and at the overview of the [supervised learning algorithms](https://scikit-learn.org/stable/supervised_learning.html#supervised-learning) offered by Scikit-learn.

Its "datasets" module also offers a quick way to load a few "toy" datasets  (i.e. simple datasets useful for education and for testing ML algorithms). Here, we use the California housing dataset from this library.

In [ ]:
# DEMO
# We load the California Housing dataset using the scikit-learn library
from sklearn.datasets import fetch_california_housing

california_housing_dataset = fetch_california_housing(as_frame=True) # Load the dataset as a pandas DataFrame

## 1.1 Global data inspection (before splitting)

Let's have a look at this dataset object

In [ ]:
# DEMO
california_housing_dataset

The dataset has a dictionary form.

Using the different keys, we can recover our inputs $X$ ('data'), our targets $\vec{y}$ ('target'), the name of each attributes for $X$ ('feature_names') and a descriptive string about the dataset ('DESCR').

In [ ]:
# DEMO
# Print the description of the dataset using the key "DESCR"
# print(california_housing_dataset["DESCR"])
# or
print(california_housing_dataset.DESCR)

From the description of this dataset, we learn that there are $m=20640$ pairs of $(\vec{x},y)$. Each $\vec{x}$ is an 8-dimensional vector representing different housing attributes, and $y$ is the median house value for a district in \\$100,000.

In [ ]:
# DEMO
# Extract the data, target and feature names
# HINT: use the keys "data", "target" and "feature_names"
X = california_housing_dataset["data"]
y = california_housing_dataset["target"]
feature_names = california_housing_dataset['feature_names']

In [ ]:
# DEMO
# Let's check what type of object is X
type(X)

As you can see, our dataset loaded from the scikit learn library is the form of **pandas DataFrames** (the pandas terminology for a table). An advantage of pandas DataFrames is that we can easily identify each column by its name, instead of its index. Usually, real life datasets are messy. There is some missing data, sometimes some wrong values and very often there is some useless data. Also, sometimes the data is not numeric: it can be the string data type, which we cannot simply feed as input to our ML model. For all these reasons, it is almost always necessary to transform the dataset before it is usable by our model, and pandas can be a useful library for that.

To learn more about pandas, work through the [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) guide.

Depending on how you load the dataset (as_frame=False or True), it can be in the form of numpy arrays as well.


Let's explore our dataset a bit more, using pandas functions.

In [ ]:
# DEMO
# Print the shape of X and y using the .shape attribute of the numpy arrays
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# DEMO
# show the first 5 rows of the DataFrame, using the head() method
X.head()

In [ ]:
# DEMO
# we can access the columns by their names, for example to get the "MedInc" column:
print(X["MedInc"])

# Or we can select multiple columns, for example "MedInc" and "HouseAge":
X[["MedInc", "HouseAge"]]

### 1.1.1 Data Cleaning
Before we can work with our dataset, we must ensure there are no missing values, no NaNs, no duplicate columns or rows

In [ ]:
# DEMO
# We can can the data types of the columns and the number of non-null values using the pandas.DataFrame.info() method
X.info()

In [ ]:
# DEMO
# We could also use the isna() method and the any() method
# Check whether there are any NaNs (same as isnull)
print("\nAny NaNs overall:", X.isna().any().any())

# Detailed missing values summary
# HINT: use the isnull() method to check for missing values, and sum() to count them
print("Missing values per column:\n", X.isnull().sum())

# Duplicate column names
dup_cols_names = X.columns[X.columns.duplicated()]
print("\nDuplicate column names:", list(dup_cols_names))

# Duplicate columns by content
dup_cols_content = X.T.duplicated()
print("\nNumber of duplicated columns by content:", dup_cols_content.sum())

# Duplicate rows
dup_rows = X.duplicated()
print("\nNumber of duplicated rows:", dup_rows.sum())

## 1.2 Splitting our dataset into a training, validation and test set

Before training our first model, we need to split the dataset into a training, a validation and a test set, so we can fit the model, tune/choose it without bias, and then get a final estimate on how it performs on new, unseen data.

- The **training set** is the subset of data used to train the model (to optimise its parameters).
- The **validation set** is used to evaluate and compare different candidate models and then select the model performing the best on this dataset. 
- The **test set** is used to estimate the performance of our model on unseen data.

We cannot use the validation set anymore for an unbiased estimate of the performance of our model. If we have evaluated multiple models on our validation subset, it is possible that the chosen model was simply ‘lucky’ on that validation data, which would lead to an overestimation of its true performance. To have a final independent evaluation, we therefore use the test set.

The ratio of data samples allocated to each of the training/validation/test set depends on the problem and the quantity of data available in total. In general, we want as much as possible data for training, while keeping enough data in the validation and test set so that the evaluation of the models stays statistically meaningful.

In the case of our housing dataset, we have about 20 000 data samples, a good data splitting ratio is 80% / 10% / 10%.


#### On sklearn's train_test_split

Scikit-learn has a function, train_test_split, that makes it easy to split a dataset into two parts. If we use it twice, we can create three datasets: training, validation, and test. By default, it shuffles the data before splitting (shuffle=True), so the split is random (use random_state if you want the same split each time).

Shuffling is important because it helps ensure that each subset is a good “mix” of the original dataset rather than being influenced by the original row order. This makes evaluation more reliable. For example, if the dataset were ordered in terms of house prices and you took the last 10% of samples as your test set, your test set might contain mostly expensive districts—so it would not represent the full range of housing districts in California.

However, for other dataset types, if samples are correlated (e.g., time series or multiple samples from the same user/patient), you should use a split strategy that respects those dependencies (for example, group-based or time-based splits).


Read the [documentation of train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to see all its options.

In [ ]:
# DEMO
from sklearn.model_selection import train_test_split

X_train, X_valtest, y_train, y_valtest = train_test_split(X, y, test_size=0.2, random_state=random_state) # using random_state for reproducibility
X_val, X_test, y_val, y_test = train_test_split(X_valtest, y_valtest, test_size=0.5, random_state=random_state) # using random_state for reproducibility

print("# of data samples for training:", len(X_train))
print("# of data samples for validation:", len(X_val))
print("# of data samples for testing:", len(X_test))

## 1.3 Exploratory data analysis (EDA)

Before training our first model, we should explore the data beyond reading its description. At minimum, we should look at **summary statistics** of the dataset—especially for the **target variable** (the output, `y`). Understanding the distribution and typical values of the target can guide model choice and evaluation.

Thus, we perform this exploration **only on the training set**. If we use the validation or test set to guide decisions, we leak information from data that is supposed to remain unseen.

We will begin by computing summary statistics for `y_train`.

### 1.3.1 First order statistics

In [ ]:
# DEMO
# Summary statistics of y_train, using pandas.DataFrame.describe()
y_train_df = pd.DataFrame(y_train, columns=["MedHouseVal"])
y_train_df.describe()

So, there is some district where the median housing price is only \$15 000... That is quite cheap. Also, the maximum median housing price for a district is only \$500 000, that is not that much (we are talking about California here, so that includes Hollywood, LA, San Francisco, etc...) but let's remember that this dataset is quite old, being from 1990. The average housing price is almost in the middle of the two extremum values, so maybe our data is normally distributed ? To verify this we will need to do a plot.

In [ ]:
# DEMO
# Summary statistics of X_train, using pandas.DataFrame.describe() method
X_train.describe()


### 1.3.2 Visualizing data

For plotting your data, you won't escape the library [matplotlib](https://matplotlib.org/3.5.0/tutorials/introductory/pyplot.html). Let's plot a simple histogram:

In [ ]:
# DEMO
import matplotlib.pyplot as plt

plt.hist(y_train, 20);

We see that the data is not really normally distributed, it is skewed to the right here. We can also see that there is an anormally high frequency of prices at about \$500 000. The maximum value for the prices is also suspect because it exactly \$500 000. This indicates that there is a maximum threshold that was applied to the dataset.

## 1.4 Model training


### 1.4.1 Linear Regression

We will begin with the simplest form of ML model for a regression, which is the linear regression: $\hat{y} = \vec{w} \cdot \vec{x} + b$, where $\vec{w}$ and $b$ are the parameters of the model. 

ML algorithms optimise their parameters with a dataset by minimising a **loss function**, i.e. a measure of the distance between the estimations made by the model $\vec{\hat{y}}$ and the true values $\vec{y}$. In the case of a linear regression, we select the parameters $\vec{w}$ and $b$ that minimise the mean squared error (mse) between $\vec{\hat{y}}$ and $\vec{y}$.

After importing the linear regression model from scikit-learn, we need to create an instance of it.

In [ ]:
# DEMO
from sklearn.linear_model import LinearRegression

# Create the Linear Regression model
lin_reg = LinearRegression()

### 1.4.2 Training a model

Then, to train the model, we only need to call the method "fit" and pass the training dataset like this:

In [ ]:
# DEMO
# Fit the model to the training data
lin_reg.fit(X_train, y_train)

It is as simple as that, and this is the same for all ML models (called "estimators" in the scikit-learn terminology) from the scikit-learn library. Just keep in mind that this simple function call hides a complex optimization or search procedures, in order to find the right parameters of the ML algorithm.

We can obtain the parameters found after training like this:

In [ ]:
# DEMO
# Get the learned parameters
w = lin_reg.coef_
b = lin_reg.intercept_

# Print the learned parameters
print("w = ", w)
print("b=", b)

### 1.4.3 Regularization: Ridge and Lasso

Plain linear regression finds the coefficients $\vec{w}$ that minimize the mse on the training data, without any constraint on their size. When features are highly correlated (multicollinearity) or numerous compared to the number of training samples, this can lead to large, unstable coefficients and a model that overfits the training data.

**Regularization** addresses this by adding a penalty on the size of the coefficients to the loss function, so the model is encouraged to keep them small:

- **[Ridge regression (L2)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)**: adds the sum of squared coefficients ($\sum w_j^2$) to the loss. This shrinks coefficients toward zero, but rarely sets them to exactly zero.
- **[Lasso regression (L1)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)**: adds the sum of absolute coefficients ($\sum |w_j|$) to the loss. This can shrink some coefficients to *exactly* zero, effectively performing feature selection.

Both models have a hyperparameter `alpha` that controls the strength of the penalty: `alpha=0` recovers plain linear regression, while increasing `alpha` increases the amount of shrinkage (more bias, less variance).

**Note on scaling**: because Ridge and Lasso penalize the *size* of coefficients, they are sensitive to the scale of the features. We have not scaled our features so far in this notebook, so treat the comparison below as a conceptual illustration rather than a properly tuned model — we'll introduce `StandardScaler` in later notebooks.

## 1.5 Model evaluation on validation set

### 1.5.1 Making new predictions with a trained model and evaluating it

We can now evaluate our model on the validation dataset. To do so, we first make the estimations $\vec{\hat{y}}_{validation}$ and then we compute the mse between it and the true values $\vec{y}_{validation}$

In [ ]:
# DEMO
# Make predictions on the validation set
y_pred_val = lin_reg.predict(X_val)

In [ ]:
# DEMO
# we can load a function to compute the mse
from sklearn.metrics import mean_squared_error

# Compute the MSE on the validation set
lin_reg_mse_val = mean_squared_error(y_val, y_pred_val) # always use y_true first, y_pred second, important for metrics that are not symmetric
lin_reg_mse_val

The mean squared error (MSE) is hard to interpret because it is in squared units. A more interpretable metric is the root mean squared error (RMSE), which is in the same units as the target. To understand whether the RMSE is “large” or “small,” we compare it to a simple reference scale, such as the average value of $\vec{y}_{validation}$.

If we were less lazy, or simply forgot about the existence of the `mean_squared_error` function, we could also compute easily the mse using numpy:

In [ ]:
float(np.mean((y_pred_val - y_val) ** 2))

In [ ]:
# DEMO
# Calculate the RMSE on the validation data, based on our MSE calculation
lin_reg_rmse_val = np.sqrt(lin_reg_mse_val)
# Calculate the mean of the true target values in the validation set, for comparison
y_val_mean = np.mean(y_val)

print("Linear Regression performance on validation data:")
print("rmse_val =", lin_reg_rmse_val)
print("y_val_mean =", y_val_mean)
print("rmse_val / y_val_mean =", lin_reg_rmse_val/y_val_mean)

### 1.5.2 Comparing our result with a heuristic or baseline model

So, on average, the housing price for the validation dataset is about \$213 000 and the estimated RMSE is \$73 000. To put this result into perspective, a good practice—before trying more complex models—is to compare it to a simple **heuristic baseline**. 

For example, we could choose $\hat{y} = mean(\vec{y}_{training})$, it is simple model that outputs a constant, the average value of the output values in the training dataset. This may sound ridiculously simple, but sometimes you can spend hours trying to find the best ML algorithm and actually end up with a result that is not much better than a simple heuristic. This can happen when the relation between the inputs and target is weak, the features are noisy or incomplete, or the labels contain substantial noise.

Depending on the distribution of $y$, the median may be a better heuristic than the mean, so we will also verify the performance of the median.

In [ ]:
# DEMO
# Heuristic-mean-model: MSE and RMSE on validation set
y_train_mean = np.mean(y_train)
y_pred_mean_val = np.full_like(y_val, y_train_mean)
heuristic_mean_mse_val = np.mean((y_pred_mean_val - y_val) ** 2)
heuristic_mean_rmse_val = np.sqrt(heuristic_mean_mse_val)

# Heuristic-median-model: MSE and RMSE on validation set
y_train_median = np.median(y_train)
y_pred_median_val = np.full_like(y_val, y_train_median)
heuristic_median_mse_val = np.mean((y_pred_median_val - y_val) ** 2)
heuristic_median_rmse_val = np.sqrt(heuristic_median_mse_val)

print("Validation MSE of different models:")
print(f"Linear Regression: {lin_reg_mse_val}")
print(f"Heuristic mean:    {heuristic_mean_mse_val}")
print(f"Heuristic median:  {heuristic_median_mse_val}")

print("\nValidation RMSE of different models:")
print(f"Linear Regression: {lin_reg_rmse_val}")
print(f"Heuristic mean:    {heuristic_mean_rmse_val}")
print(f"Heuristic median:  {heuristic_median_rmse_val}")

Thoses results are clearly showing that the linear model is performing better than the heuristics, and the linear model being still a very simple model, we probably can try a more complex model. But before doing that, you can compare the training and validation loss value, which can help us get an idea whether we are **underfitting** or **overfitting**.

-  **Overfitting** is a problem that you are more likely to experience if you use a ML model that is too complex (with many parameters) compared to the amount of training data samples (too few). 
- **Underfitting** happens if you choose a model that is too simple to represent the (more complex) relationships present in the training data available (a lot of data). 

A much better performance on the training set than on the validation set, is an indicator of overfitting. On the other hand, if the performance is below expectations both on training and validation set, your model might be underfitting.

In [ ]:
# DEMO
y_pred_linreg_train = lin_reg.predict(X_train)
lin_reg_mse_train = np.mean((y_pred_linreg_train - y_train) ** 2)

print("training loss:", lin_reg_mse_train)
print("validation loss:", lin_reg_mse_val)

Would we be overfitting, underfitting or just right?

Try to re-run the notebook after changing the random seed and the random state, and check the new loss values found. You will then find different values! To obtain a more reliable (lower variance) of our validation loss, we would need to increase the size of the validation dataset, but we do not want to reduce the training dataset size either... The solution in those cases is using **cross-validation**.

### 1.5.4 Cross-validation

In **cross-validation**, we combine the training and validation data and split it into $n$ folds, typically $2 \leq n \leq 10$. We then train the model on 
n−1 folds and evaluate it on the remaining fold. We repeat this procedure so each fold serves once as the validation set. Finally, we average the n validation losses to obtain a more reliable estimate of performance, based on more validation samples.

Scikit-learn offers different ways to perfrom cross validation. Here, we will just use a function to help us split our training and validation data into 5 folds, then do the cross validation ourself, to obtain a better understanding of the concept.

In [ ]:
# DEMO
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# we create a 5 folds splitting object that will return the training and validation indexes for our 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=random_state) # we shuffle the data before splitting into folds so that each fold is more representative of the overall data

# We merge train and validation sets (the test set remains untouched)
X_trainval = np.concatenate((X_train, X_val))
y_trainval = np.concatenate((y_train, y_val))

# Lists to save the losses
train_mses = []
val_mses = []

# loop through the different folds
for train_indexes, val_indexes in kf.split(X_trainval):
    
    # create an new linreg model, untrained
    lin_reg_temp = LinearRegression()
    # train the model on a subset of the training data
    lin_reg_temp.fit(X_trainval[train_indexes], y_trainval[train_indexes])
    
    # make prediction for training and validation dataset
    y_pred_train_temp = lin_reg_temp.predict(X_trainval[train_indexes])
    y_pred_val_temp = lin_reg_temp.predict(X_trainval[val_indexes])
    
    # compute the losses
    mse_train_temp = mean_squared_error(y_trainval[train_indexes], y_pred_train_temp) # Metrics always take (y_true, y_pred), in this order
    mse_val_temp = mean_squared_error(y_trainval[val_indexes], y_pred_val_temp) # Each validation loss is computed on data that was not used to train the model in that fold
    
    # append the loss values to the lists
    train_mses.append(mse_train_temp)
    val_mses.append(mse_val_temp)
    
print(f"Mean training MSE: {round(np.mean(train_mses), 3)} (std: {round(np.std(train_mses), 3)})")
print(f"Mean validation MSE: {round(np.mean(val_mses), 3)} (std: {round(np.std(val_mses), 3)})")

The two loss values are very close, with the validation loss bigger than the training loss, which is expected. In the case of underfitting, we expect our training and validion loss to be close together and 'high', so it is likely that our simple model is underfitting. 

Presuming that we are underfitting, we can try more complex non-linear models. It would be a good exercise to try at least 2 other more complex models, compare their performance on the validation dataset and try to tell if they are overfitting or not. After selecting a model based on the the cross-validation results, we must retrain the model on the whole training+validation set:

In [ ]:
# DEMO
# Retrain the model on the whole training+validation set
lin_reg_final = LinearRegression()
lin_reg_final.fit(X_trainval, y_trainval)

To compare the performance of different models, you usually should not restrict yourself to only compare their respective loss values (the mse in our current regression problem). You can compute a multitude of **evaluation metrics** that can help you identify some problem in the performance of your model, that you would not be able to detect if you were only looking at the loss value otherwise. Scikit-learn allows you to compute [a few popular metrics](https://scikit-learn.org/stable/modules/model_evaluation.html).

## 1.6 Estimate performance on test set

Let's say we have chosen the linear regression model over the heuristics, based on performance on the validation set. Since we used the validation set to choose the best model
, we need the test set to estimate the performance on **unseen data**.

### 1.6.1 Calculate performance metrics

In [ ]:
# DEMO
# Compute metrics of the lin_reg_final model on test set
y_pred_linreg_test = lin_reg_final.predict(X_test)
lin_reg_mse_test = mean_squared_error(y_test, y_pred_linreg_test)
lin_reg_rmse_test = np.sqrt(lin_reg_mse_test)

# Print test set MSE and RMSE
print("Test MSE:", lin_reg_mse_test)
print("Test RMSE:", lin_reg_rmse_test)

We again have a similar, but slightly higher MSE than on the training folds. Is this an expected result?

# 2. Classification mini-project

After looking at a regression example, we will now turn to a **classification problem**. In classification, the target is not a continuous value but a **discrete label** (a class). As an example, we will use the Iris dataset and try to predict flower types based on a couple of features. 

We will use a **logistic regression** model. If the model is new to you, start with this [intro to logistic regression](https://www.geeksforgeeks.org/machine-learning/understanding-logistic-regression/).

Read the [documentation on logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) in scikit-learn.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
# EXERCISE
# Load the Iris dataset as a pandas DataFrame
# HINT: iris_ds is the variable name for the dataset object
# HINT: df_iris is the variable name for the DataFrame (with data and target), use .frame
iris_ds = load_iris(as_frame=True)
df_iris = iris_ds.frame 

## 2.1 Global data inspection (before splitting)

In [ ]:
# EXERCISE
# Print the description of the dataset
# HINT: use iris_ds and the key "DESCR"
print(iris_ds["DESCR"])
# or
print(iris_ds.DESCR)

As you can see from the description, the output data can take 3 different values, which are 3 types of flowers. Those three classes are Iris-Setosa, Iris-Versicolour and Iris-Virginica. The input data has 4 attributes, which are the length and width measurements of the sepal and petal of each flower, and there are measurements for 150 flowers in the dataset. 

In [ ]:
# EXERCISE
# Print the shape of the DataFrame (rows, columns)
# HINT: use the shape attribute of the DataFrame
print("Shape (rows, columns):", df_iris.shape)

# Print the first 5 rows of the DataFrame
# HINT: use the head() method of the DataFrame to print the first 5 rows
df_iris.head()

In [ ]:
# EXERCISE
# Extract the data and target from the iris dataset
# HINT: use the keys "data" and "target" of the iris_ds object
X = iris_ds["data"]
y = iris_ds["target"]
# or
X = iris_ds.data
y = iris_ds.target

# Print the shape of X and y
print("X shape:", X.shape)
print("y shape:", y.shape)

### 2.1.1 Data Cleaning
Before we can work with our dataset, we must ensure there are no missing values, no NaNs, no duplicate columns or rows

In [ ]:
# EXERCISE
# Check for missing values in the classification dataset
# Option: using .isnull to check for missing values 
print("Missing values per feature:", X.isnull().sum(axis=0))
print("Any NaNs in X:", X.isnull().any().any())
print("Any NaNs in y:", y.isnull().any())

# Option: using .info to check for missing values
print("==="*20)
X.info()
print("==="*20)
y.info()


In [ ]:
# EXERCISE
# Check for duplicate rows in X
# HINT: use the duplicated() method of the DataFrame, and sum() to count
int(X.duplicated().sum())

Now we were only looking at  our features. Let's see is this is a duplicate sample, by checking the whole dataframe with both features (inputs) and target: df_iris.

In [ ]:
# EXERCISE
# Check for duplicate samples in our dataset, using pandas.DataFrame.duplicated() method
# HINT: use the duplicated() method of the DataFrame, and sum() to count
num_duplicated_samples_df = df_iris.duplicated().sum()
print("Number of duplicated samples (X, y):", num_duplicated_samples_df)

We indeed do have a duplicate sample. We don't know if 2 flowers ended up having the exect same measurement, or one sample was added twice due to a human error. It could cause snooping if the 2 versions of the sample end up in different folds of our dataset, so we delete the duplicate to be safe.

In [ ]:
# EXERCISE
# Remove duplicate samples from our dataset, using pandas.DataFrame.drop_duplicates() method
# HINT: use the drop_duplicates() method of the DataFrame, and check the shape
df_iris_no_duplicates = df_iris.drop_duplicates()
print("Shape of the DataFrame after removing duplicates:", df_iris_no_duplicates.shape)

In [ ]:
# EXERCISE
# Extract the data and target from the DataFrame without duplicates, to use for ML training
X = df_iris_no_duplicates.drop(columns=["target"])   # DataFrame
y = df_iris_no_duplicates["target"]                  # Series


### 2.1.2 Checking the class distribution

In a classification problem, it is important to check the **class distribution**; whether there is **class imbalance**.

In this dataset, the three classes are balanced, there are 50 appearances for each of the 3 classes. In many real-world datasets, however, some classes are under-represented. Models trained on such imbalanced data might perform poorly on the minority class. Also, standard metrics such as [accuracy](https://scikit-learn.org/stable/modules/model_evaluation.html#accuracy-score) - defined as the fraction of correct predictions - can become misleading. A model could for example achieve high accuracy by mostly predicting the majority class. Other metrics such as precision, recall and the [F1 score](https://en.wikipedia.org/wiki/F-score) are more informative then.

In [ ]:
# DEMO
# If the class distribution was not given yet in the dataset description, we could compute it e.g. as follows:
unique, counts = np.unique(y, return_counts=True)
class_distribution = dict(zip(iris_ds["target_names"][unique], counts))
print("Class distribution:")
for cls, cnt in class_distribution.items():
    print(f"  {cls}: {cnt} samples")

When dealing with categorical data, the data is usually represented as an integer value $k=0,...,K-1$ where K the number of different category/class existing. Sometimes the data has a string format, and the value have the name of the category. When this is the case, we first need to convert the classes from string to an integer value. As you can see below, the values of $y$ are already integer.

In [ ]:
# DEMO
y

## 2.2 Splitting the dataset into training and test set

There are only 150 data samples, so we will use **cross-validation** instead of splitting our dataset into a training, a validation and test set. We keep 50 data samples for testing

We **stratify** our split (using our class labels in input to tell the function how to stratify). Stratifying a split means keeping the class proportions roughly the same in each subset (train/validation/test) as in the full dataset. Like this, we ensure all classes occur in each subset. This is especially important in the case of unbalanced datasets.

Don't forget to set the random state for reproducability; to get the same split each time you run the notebook.

In [ ]:
from sklearn.model_selection import train_test_split
# EXERCISE
# Use the train_test_split function to split the data into (training+validation) and test set
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=40, random_state=random_state, stratify=y)


In [ ]:
# DEMO
# Let's check whether our stratified split performed as expected
# Check class distribution in training set
unique_train, counts_train = np.unique(y_trainval, return_counts=True)
print("Training set class distribution:")
for label, count in zip(iris_ds["target_names"][unique_train], counts_train):
    print(f"  {label}: {count} samples")

# Check class distribution in test set
unique_test, counts_test = np.unique(y_test, return_counts=True)
print("\nTest set class distribution:")
for label, count in zip(iris_ds["target_names"][unique_test], counts_test):
    print(f"  {label}: {count} samples")

## 2.3 Training set Exploratory Data Analysis (EDA)

From this point onward, we perform EDA only on the train+validation pool used for cross-validation, and we keep the test set untouched for the final evaluation. This to avoid peeking the final test set.

### 2.3.1 Summary Statistics

In [ ]:
# EXERCISE
# Summary statistics of X_trainval, using pandas.DataFrame.describe() method
X_trainval.describe()

**Another note on scaling**
Features have different scales, which can affect distance-based or gradient-based classifiers. Scikit-learn's implementation of logistic regression uses regularization by default, which also performs better with scaled features. In future Notebooks, we will therefore learn how to normalize the data.

### 2.3.2 Data Visualization

Visualizing your data helps to understand the problem your are trying to solve. We visualize the 4 features of the Iris dataset in 2 separate plots.

In [ ]:
# DEMO
# Let's visualize the Iris dataset, so we understand the problem we are trying to solve
import matplotlib.pyplot as plt

feature_names = iris_ds["feature_names"]
target_names = iris_ds["target_names"]

def scatter_by_class(ax, X, y, x_col, y_col, plot_title, target_names=None):
    # x_col / y_col can be column names (recommended) or integer positions
    if isinstance(x_col, int):
        x_col = X.columns[x_col]
    if isinstance(y_col, int):
        y_col = X.columns[y_col]

    classes = sorted(set(y))
    for c in classes:
        label = target_names[c] if target_names is not None and isinstance(c, int) else str(c)
        m = (y == c)
        ax.scatter(X.loc[m, x_col], X.loc[m, y_col], label=label, alpha=0.7)

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(plot_title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

scatter_by_class(axes[0], X, y, 0, 1, "Feature 1 vs 2", target_names=target_names)
scatter_by_class(axes[1], X, y, 2, 3, "Feature 3 vs 4", target_names=target_names)

plt.tight_layout()
plt.show()

## 2.4 Model training and validation

### 2.4.0 Baseline performance (Dummy Classifier)
Before training any real model, we estimate a baseline performance using a trivial classifier. This helps us understand what performance can be achieved without learning any meaningful relationship between features and labels. Here, we classify every instance with the most frequent class label in the training set.

In [ ]:
# DEMO
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Baseline classifier: always predicts the most frequent class
dummy_clf = DummyClassifier(strategy="most_frequent", random_state=random_state)

### 2.4.1 Logistic regression

We will use the classifier version of the linear regression model used previously: the **logistic regression**. This linear model predict a probability a score for each class, and then select the class with the highest estimated probability as the predicted class.

Read the [documentation of LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) to see all its parameters.

In [ ]:
# DEMO
from sklearn.linear_model import LogisticRegression

log_reg_clf = LogisticRegression()

### 2.4.2 Kfold cross validation

**Accuracy** measures the proportion of correctly classified samples, i.e. the fraction of predictions that match the true labels. While it is simple and intuitive, it can be misleading in problems with imbalanced classes or unequal error costs. Later on, we will therefore introduce additional performance metrics that give a more complete evaluation of classification models.

For our cross-validation here, we will stick to accuracy as our performance metric.

In [ ]:
# DEMO
# scikit-learn offers functions to compute different metrics, such as the accuracy
from sklearn.metrics import accuracy_score

When the dataset is small, it is often preferable to use cross-validation with a larger number of folds, because each model is trained on a larger fraction of the data. However, we do not always choose a high number of folds because cross-validation becomes more computationally expensive: for $k$ folds, the model must be trained 
$k$ times.

We use **StratifiedKFold** to ensure that each fold contains samples from all classes and that the class proportions are approximately the same across folds.

In [ ]:
# DEMO
# The cross-validation procedure for our dummy classifier
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_state)

dummy_train_acc = []
dummy_val_acc = []

for train_idx, val_idx in kf.split(X_trainval, y_trainval):
    dummy = DummyClassifier(strategy="most_frequent", random_state=random_state)
    dummy.fit(X_trainval.iloc[train_idx], y_trainval.iloc[train_idx])

    y_pred_train = dummy.predict(X_trainval.iloc[train_idx])
    y_pred_val   = dummy.predict(X_trainval.iloc[val_idx])

    dummy_train_acc.append(accuracy_score(y_trainval.iloc[train_idx], y_pred_train))
    dummy_val_acc.append(accuracy_score(y_trainval.iloc[val_idx], y_pred_val))

print(f"Dummy baseline - mean train acc: {np.mean(dummy_train_acc):.3f}; std: {np.std(dummy_train_acc):.3f}")
print(f"Dummy baseline - mean val acc:   {np.mean(dummy_val_acc):.3f}; std: {np.std(dummy_val_acc):.3f}")

In [ ]:
# EXERCISE
# The cross-validation procedure for our logistic regression classifier
from sklearn.linear_model import LogisticRegression

# we use the same kf object as defined before for the dummy classifier

train_folds_acc = []
val_folds_acc = []

for train_idx, val_idx in kf.split(X_trainval, y_trainval):

    log_reg_clf = LogisticRegression(random_state=random_state)
    log_reg_clf.fit(X_trainval.iloc[train_idx], y_trainval.iloc[train_idx])

    y_pred_train = log_reg_clf.predict(X_trainval.iloc[train_idx])
    y_pred_val   = log_reg_clf.predict(X_trainval.iloc[val_idx])

    train_folds_acc.append(accuracy_score(y_trainval.iloc[train_idx], y_pred_train))
    val_folds_acc.append(accuracy_score(y_trainval.iloc[val_idx], y_pred_val))

print(f"LogReg - mean train acc: {np.mean(train_folds_acc):.3f}; std: {np.std(train_folds_acc):.3f}")
print(f"LogReg - mean val acc:   {np.mean(val_folds_acc):.3f}; std: {np.std(val_folds_acc):.3f}")

Our logistic regression model performs better than the dummy baseline, which suggests it has learned a useful relationship between the features and the class labels.

The accuracy is strong on both the training and validation folds, and the two values are close to each other. This is a good sign: although small datasets can be easy to overfit with very complex models (sometimes reaching 100% training accuracy), we do not see a large gap here, so there is no clear evidence of overfitting. We also observe a relatively high variance across the validation folds, which is expected when using many folds (each validation split is small, so the score can fluctuate more). Finally, the overall accuracy is quite high—this is plausible for a simple toy dataset like Iris, but in real-world problems unusually strong performance should always be checked with basic sanity checks (e.g., data leakage, correct splitting strategy, and evaluation on a separate test set).|

## 2.5 Final evaluation on test set

Assuming we have decided to stick with logistic regression, we can now **train a final model using the combined training + validation data** (since we no longer need the validation set for model selection). We then evaluate this final model once on the test set to obtain an unbiased estimate of its performance on unseen data.

Finally, accuracy is a convenient first metric, but it is not always sufficient—especially when classes are imbalanced or when different types of errors have different costs. Scikit-learn provides many [evaluation metrics](https://scikit-learn.org/stable/modules/model_evaluation.html). For example, classification_report summarizes precision, recall, and F1-score for each class. We will study these metrics in more detail in future notebooks. For now, take a moment to look them up and make sure you understand what they measure and why they are useful.

In [ ]:
# EXERCISE
# training a model on the training-validation dataset
# if you experienced convergence warnings, you can increase the max_iter parameter of the LogisticRegression model, for example to 200
log_reg_clf = LogisticRegression(random_state=random_state, max_iter=200) 
log_reg_clf.fit(X_trainval, y_trainval)

# make prediction for the test dataset
y_pred_test = log_reg_clf.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report
# DEMO
print(classification_report(y_test, y_pred_test))


**Disclaimer**: GenAI was used in this notebook for some code sections and to improve phrasing.